# Nested CV - stabilised selection (run 2)

Companion to `nested_cv.ipynb` (run 1, kept frozen for reference). Run 1 diagnosed a
**tuning overfit**: the inner loop drifted to the most aggressive warping (penalty 0.01 in
22/50 folds) and the lowest basis count (n_basis 30 in 32/50), yet scored *lower*
(amplitude AUC 0.824) than the fixed-setting baseline (0.874, which sat above run 1's p90).
With ~53 curves per inner fold, 3-fold argmax selection was chasing noise.

This run changes **only the selection protocol**:

| | run 1 | run 2 (this) |
|---|---|---|
| inner folds | 3 | **5** |
| selection | argmax inner AUC | **1-SE rule** (Breiman et al. 1984) |
| penalty grid | [0.01, 0.05, 0.1, 0.5] | unchanged |
| n_basis grid | [30, 40, 60, 80] | unchanged |
| n_components | tuned {3,5,8,10} | **fixed at 5** |
| C | tuned {0.01,0.1,1,10} | **fixed at 1.0** |
| repeats | 10 | 10 (apples-to-apples) |

**Why those two axes are fixed, and fixed *there*.** Both values are carried over from the
fixed-setting baseline run that produced AUC 0.874 - they are *not* read off run 1's selection
histograms. Picking constants out of run 1's histograms would be a post-hoc, test-informed
choice (we have already seen run 1's outer AUC), so it would quietly reimport the optimism
nested CV exists to remove. Continuity with a pre-existing decision is defensible; histogram
mining is not. This leaves only the two axes that govern the FDA representation
(penalty, n_basis) under tuning - a 16-cell grid instead of 256, which is itself a large cut
in the selection freedom that caused the overfit.

**Grid floors are unchanged on purpose.** n_basis stays floored at 30: over the 2 s window
that is ~67 ms knot spacing, and a spike is 20-70 ms / a sharp wave 70-200 ms. Extending
downward (15 -> ~133 ms) would let the model win by smoothing away the very sharp transient
whose shape is the scientific claim. Penalty stays capped at 0.5 because the 1-SE rule needs
the low-warping end to back off *into*.

**Honesty caveat, to carry into the writeup:** this protocol was revised after seeing run 1's
outer AUC. Run 2's number is therefore a *sensitivity analysis of the selection rule*, not a
fresh unbiased headline. Report it as such.

In [6]:
import sys, time
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from skfda import FDataGrid
from skfda.preprocessing.dim_reduction import FPCA
from skfda.preprocessing.registration import FisherRaoElasticRegistration
from skfda.preprocessing.smoothing import BasisSmoother
from skfda.representation.basis import BSplineBasis

sys.path.insert(0, str(Path('..') / 'src'))
from classify_loader import load_CSV, load_edf

plt.style.use('../imagestyle.mplstyle')
ROOT    = Path('..')
CSV     = ROOT / 'Kural_Dataset' / 'eeg_summary.csv'
EDF_DIR = ROOT / 'Kural_Dataset' / 'Recordings'

In [7]:
manifest = load_CSV(CSV)
SFREQ     = 500.0
HALF_N    = int(1.0 * SFREQ)
PEAK_HALF = int(round(0.1 * SFREQ))

curves, labels, certs = [], [], []
for _, row in manifest.iterrows():
    raw   = load_edf(row['file_id'], EDF_DIR)
    onset = int(round(row['transient_onset_s'] * SFREQ))
    data  = raw.get_data(start=onset - HALF_N, stop=onset + HALF_N)
    ch    = np.argmax(np.ptp(data[:, HALF_N - PEAK_HALF: HALF_N + PEAK_HALF], axis=1))
    curves.append(data[ch])
    labels.append(row['label_binary'])
    certs.append(row['certainty'])

t    = (np.arange(2 * HALF_N) - HALF_N) / SFREQ
fd   = FDataGrid(data_matrix=np.stack(curves), grid_points=t)
y    = np.array(labels)
cert = np.array(certs)
print(f'Loaded {len(y)} curves  (epileptic={y.sum()}, non-epileptic={(y==0).sum()})')
for g, c in zip(*np.unique(cert, return_counts=True)):
    print(f'  {g}: {c}')

Loaded 100 curves  (epileptic=54, non-epileptic=46)
  clear_positive: 22
  negative: 46
  unclear_positive: 32


## Pipeline

Unchanged from run 1. `SmoothRegister` (B-spline smooth -> resample -> Fisher-Rao) depends only
on `(n_basis, penalty)` and is leakage-safe: `fit` learns the Karcher-mean template from
training curves only, `transform` aligns new curves to that stored template.

In [8]:
class SmoothRegister(BaseEstimator, TransformerMixin):
    """B-spline smooth -> resample -> Fisher-Rao registration. Depends only on (n_basis, penalty)."""
    def __init__(self, n_basis=40, penalty=0.1, n_reg_points=100):
        self.n_basis      = n_basis
        self.penalty      = penalty
        self.n_reg_points = n_reg_points

    def fit(self, X, y=None):
        domain = X.domain_range[0]
        self._smoother = BasisSmoother(
            BSplineBasis(domain_range=domain, n_basis=self.n_basis, order=4),
            return_basis=False)
        X_sm    = self._smoother.fit_transform(X)
        self._t = np.linspace(domain[0], domain[1], self.n_reg_points)
        self._reg = FisherRaoElasticRegistration(penalty=self.penalty)
        self._reg.fit(FDataGrid(X_sm(self._t).squeeze(-1), self._t))
        return self

    def transform(self, X):
        X_sm = self._smoother.transform(X)
        X_c  = FDataGrid(X_sm(self._t).squeeze(-1), self._t)
        X_al = self._reg.transform(X_c)
        return {
            'aligned': X_al.data_matrix[:, :, 0],
            'warps':   self._reg.warping_.data_matrix[:, :, 0],
            't':       self._t,
        }


class FPCAFeatures(BaseEstimator, TransformerMixin):
    """FPCA on amplitude, phase, or both. Receives dict from SmoothRegister."""
    def __init__(self, n_components=5, mode='amplitude'):
        self.n_components = n_components
        self.mode         = mode

    def fit(self, X, y=None):
        t = X['t']
        if self.mode in ('amplitude', 'both'):
            self._fpca_amp = FPCA(n_components=self.n_components)
            self._fpca_amp.fit(FDataGrid(X['aligned'], t))
        if self.mode in ('phase', 'both'):
            self._fpca_ph = FPCA(n_components=self.n_components)
            self._fpca_ph.fit(FDataGrid(X['warps'], t))
        return self

    def transform(self, X):
        t = X['t']
        feats = []
        if self.mode in ('amplitude', 'both'):
            feats.append(self._fpca_amp.transform(FDataGrid(X['aligned'], t)))
        if self.mode in ('phase', 'both'):
            feats.append(self._fpca_ph.transform(FDataGrid(X['warps'], t)))
        return np.hstack(feats)

## Configuration

Two tuned axes (16 cells), two fixed axes carried over from the fixed-setting baseline.

In [9]:
# --- tuned axes (unchanged from run 1) ---
PENALTIES = [0.01, 0.05, 0.1, 0.5]   # spans the responsive region of the penalty sweep
N_BASES   = [30, 40, 60, 80]         # floor 30 = ~67 ms knots, still spike-resolving

# --- fixed axes: values from the fixed-setting baseline run (AUC 0.874), not from run 1 histograms ---
N_COMPONENTS = 5
C_FIXED      = 1.0

N_REPEATS = 10
OUTER_K   = 5
INNER_K   = 5        # was 3 in run 1 - the main stabilisation
MODES     = ['amplitude', 'phase', 'both']
N_JOBS    = -1

# Reference numbers to compare against (both at 10 repeats where applicable)
RUN1_NESTED = {'amplitude': 0.824, 'phase': 0.615, 'both': 0.823}   # 3-fold inner, argmax
FIXED_AUC   = {'amplitude': 0.874, 'phase': 0.560, 'both': 0.857}   # single-loop, fixed settings


def compute_metrics(y_true, y_prob, cert):
    auc      = roc_auc_score(y_true, y_prob)
    y_pred   = (y_prob >= 0.5).astype(int)
    spec     = np.mean(y_pred[cert == 'negative']         == 0)
    sens_clr = np.mean(y_pred[cert == 'clear_positive']   == 1)
    sens_unc = np.mean(y_pred[cert == 'unclear_positive'] == 1)
    return {'auc': auc, 'specificity': spec, 'sens_clear': sens_clr, 'sens_unclear': sens_unc}


print(f'Tuned grid: {len(PENALTIES) * len(N_BASES)} cells (penalty x n_basis)')
print(f'Fixed: n_components={N_COMPONENTS}, C={C_FIXED}')
print(f'Registrations per outer fold: {len(PENALTIES) * len(N_BASES) * INNER_K}')

Tuned grid: 16 cells (penalty x n_basis)
Fixed: n_components=5, C=1.0
Registrations per outer fold: 80


## Nested CV with the 1-SE rule

**1-SE rule.** For each of the 16 cells, take the mean inner-fold AUC and its standard error
across the 5 inner folds. Find the best mean, then keep every cell whose mean is within 1 SE
*of the best cell*. Among those statistically-tied candidates, take the **simplest** model
rather than the argmax. This is what counters the noise-chasing seen in run 1.

**"Simplest" is defined explicitly**, because the direction matters:
- **higher penalty first** - less warping, a more constrained registration. This is the axis
  where run 1's drift was diagnosed (it kept picking 0.01), so this is where the rule does its work.
- **fewer basis as tie-break** - bounded below by the grid floor of 30, so it cannot smooth
  the transient away.

The registration cache is shared across the three modes (splits are seed-identical and
registration is mode-independent), and repeats run in parallel - same optimisations as run 1.

We also record the plain argmax pick alongside the 1-SE pick, to quantify how far the rule
actually backed off.

In [10]:
def feats(entry, mode, split):
    """Assemble the feature block for a mode from cached FPCA scores."""
    amp, ph = entry[f'amp_{split}'], entry[f'phase_{split}']
    if mode == 'amplitude':
        return amp
    if mode == 'phase':
        return ph
    return np.hstack([amp, ph])


def select_1se(means, ses):
    """Standard 1-SE rule over the (penalty, n_basis) grid.
    Among cells within 1 SE of the best mean inner AUC, return the simplest:
    highest penalty (least warping) first, fewest basis as tie-break."""
    best   = max(means, key=means.get)
    thresh = means[best] - ses[best]
    cands  = [k for k, m in means.items() if m >= thresh]
    return min(cands, key=lambda k: (-k[0], k[1])), best


def process_repeat(rep):
    n      = len(y)
    oof    = {m: np.zeros(n) for m in MODES}
    params = {m: [] for m in MODES}
    outer_cv = StratifiedKFold(n_splits=OUTER_K, shuffle=True, random_state=rep)

    for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(np.arange(n), cert)):
        y_tr, cert_tr = y[tr_idx], cert[tr_idx]
        inner_splits = list(
            StratifiedKFold(n_splits=INNER_K, shuffle=True, random_state=rep * 100 + fold)
            .split(np.arange(len(tr_idx)), cert_tr)
        )

        # Registration + FPCA cache: built ONCE per outer fold, reused by all three modes.
        # 16 (pen, nb) x 5 inner folds = 80 registrations per outer fold.
        cache = {}
        for pen, nb in product(PENALTIES, N_BASES):
            for i_fold, (i_tr, i_te) in enumerate(inner_splits):
                sr     = SmoothRegister(n_basis=nb, penalty=pen)
                reg_tr = sr.fit_transform(fd[tr_idx[i_tr]])
                reg_te = sr.transform(fd[tr_idx[i_te]])
                t_reg  = reg_tr['t']
                entry  = {'y_tr': y_tr[i_tr], 'y_te': y_tr[i_te]}
                for key, field in [('amp', 'aligned'), ('phase', 'warps')]:
                    fpca = FPCA(n_components=N_COMPONENTS)
                    fpca.fit(FDataGrid(reg_tr[field], t_reg))
                    entry[f'{key}_tr'] = fpca.transform(FDataGrid(reg_tr[field], t_reg))
                    entry[f'{key}_te'] = fpca.transform(FDataGrid(reg_te[field], t_reg))
                cache[(pen, nb, i_fold)] = entry

        for mode in MODES:
            # Inner sweep: mean and SE of inner-fold AUC for each of the 16 cells
            means, ses = {}, {}
            for pen, nb in product(PENALTIES, N_BASES):
                aucs = []
                for i_fold in range(INNER_K):
                    e  = cache[(pen, nb, i_fold)]
                    sc = StandardScaler()
                    lr = LogisticRegression(C=C_FIXED, max_iter=1000, class_weight='balanced')
                    lr.fit(sc.fit_transform(feats(e, mode, 'tr')), e['y_tr'])
                    aucs.append(roc_auc_score(
                        e['y_te'],
                        lr.predict_proba(sc.transform(feats(e, mode, 'te')))[:, 1]))
                means[(pen, nb)] = np.mean(aucs)
                ses[(pen, nb)]   = np.std(aucs, ddof=1) / np.sqrt(INNER_K)

            (pen, nb), (a_pen, a_nb) = select_1se(means, ses)

            # Refit the 1-SE pick on the full outer training set, predict the held-out fold
            sr     = SmoothRegister(n_basis=nb, penalty=pen)
            reg_tr = sr.fit_transform(fd[tr_idx])
            fp     = FPCAFeatures(n_components=N_COMPONENTS, mode=mode)
            sc     = StandardScaler()
            lr     = LogisticRegression(C=C_FIXED, max_iter=1000, class_weight='balanced')
            lr.fit(sc.fit_transform(fp.fit_transform(reg_tr)), y_tr)

            reg_te = sr.transform(fd[te_idx])
            oof[mode][te_idx] = lr.predict_proba(sc.transform(fp.transform(reg_te)))[:, 1]
            params[mode].append({'penalty': pen, 'n_basis': nb,
                                 'argmax_penalty': a_pen, 'argmax_n_basis': a_nb,
                                 'backed_off': (pen, nb) != (a_pen, a_nb)})
        del cache

    return oof, params


t0 = time.time()
rep_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(process_repeat)(rep) for rep in range(N_REPEATS)
)

ncv_results = {m: [] for m in MODES}
ncv_params  = {m: [] for m in MODES}
for oof, params in rep_results:
    for m in MODES:
        ncv_results[m].append(compute_metrics(y, oof[m], cert))
        ncv_params[m].extend(params[m])

print(f'\nDone in {(time.time() - t0) / 60:.1f} min')
for m in MODES:
    aucs = [r['auc'] for r in ncv_results[m]]
    print(f'  [{m}] AUC {np.mean(aucs):.3f}  (across {N_REPEATS} repeats)')

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of  10 | elapsed: 35.8min remaining: 83.6min
[Parallel(n_jobs=-1)]: Done   5 out of  10 | elapsed: 35.9min remaining: 35.9min
[Parallel(n_jobs=-1)]: Done   7 out of  10 | elapsed: 35.9min remaining: 15.4min



Done in 48.5 min
  [amplitude] AUC 0.857  (across 10 repeats)
  [phase] AUC 0.606  (across 10 repeats)
  [both] AUC 0.827  (across 10 repeats)


[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed: 48.5min finished


## Results

In [11]:
metrics    = ['auc', 'specificity', 'sens_clear', 'sens_unclear']
met_labels = ['AUC', 'Specificity', 'Sens clear+', 'Sens unclear+']

rows = []
for mode in MODES:
    row = {'Condition': mode}
    for m, ml in zip(metrics, met_labels):
        vals = [r[m] for r in ncv_results[mode]]
        mn, p10, p90 = np.mean(vals), np.percentile(vals, 10), np.percentile(vals, 90)
        row[ml] = f'{mn:.3f}  ({p10:.3f}-{p90:.3f})'
    rows.append(row)

print(pd.DataFrame(rows).set_index('Condition').to_string())
print()
print('AUC comparison (all nested numbers at 10 repeats - apples to apples):')
print(f'{"condition":12s} {"run2 (5f,1SE)":>14s} {"run1 (3f,argmax)":>17s} {"fixed-setting":>14s}')
for mode in MODES:
    r2 = np.mean([r['auc'] for r in ncv_results[mode]])
    print(f'{mode:12s} {r2:>14.3f} {RUN1_NESTED[mode]:>17.3f} {FIXED_AUC[mode]:>14.3f}'
          f'   (vs run1 {r2 - RUN1_NESTED[mode]:+.3f})')

                            AUC           Specificity           Sens clear+         Sens unclear+
Condition                                                                                        
amplitude  0.857  (0.822-0.881)  0.854  (0.826-0.891)  0.814  (0.768-0.864)  0.703  (0.650-0.781)
phase      0.606  (0.537-0.672)  0.578  (0.520-0.680)  0.664  (0.491-0.832)  0.500  (0.403-0.594)
both       0.827  (0.789-0.859)  0.807  (0.800-0.826)  0.805  (0.768-0.868)  0.634  (0.525-0.750)

AUC comparison (all nested numbers at 10 repeats - apples to apples):
condition     run2 (5f,1SE)  run1 (3f,argmax)  fixed-setting
amplitude             0.857             0.824          0.874   (vs run1 +0.033)
phase                 0.606             0.615          0.560   (vs run1 -0.009)
both                  0.827             0.823          0.857   (vs run1 +0.004)


## Selected settings + the registration-collapse guard

Two things to check here.

1. **Did the 1-SE rule actually do anything?** `backed_off` counts the folds where it chose a
   simpler cell than the plain argmax. If that is near zero, the argmax was already at the
   simple end and 1-SE is not the lever - any AUC change came from the 5-fold inner CV instead.

2. **Did registration get switched off?** "Simpler" on the penalty axis means *less warping*,
   and penalty 0.5 gives `mean_max_dev` ~0.056 s - warps collapsing toward identity. If 1-SE
   pushes most folds to 0.5, the "registered" arm has quietly become an unregistered one and
   the comparison to `classification.ipynb` is confounded. The guard below flags that.

In [12]:
for mode in MODES:
    df_p = pd.DataFrame(ncv_params[mode])
    n_f  = len(df_p)
    print(f'\n=== {mode}  ({n_f} outer folds) ===')
    print(f'  1-SE penalty : {dict(df_p["penalty"].value_counts().sort_index())}')
    print(f'  1-SE n_basis : {dict(df_p["n_basis"].value_counts().sort_index())}')
    print(f'  argmax penalty (what run 1 would have picked): '
          f'{dict(df_p["argmax_penalty"].value_counts().sort_index())}')
    print(f'  backed off from argmax: {df_p["backed_off"].sum()}/{n_f} folds')

    frac_max_pen = np.mean(df_p['penalty'] == max(PENALTIES))
    if frac_max_pen > 0.5:
        print(f'  !! WARNING: {frac_max_pen:.0%} of folds sit at penalty={max(PENALTIES)} '
              f'(warps ~= identity). Registration is effectively OFF for this condition - '
              f'the registered/unregistered comparison is confounded.')
    else:
        print(f'  OK: only {frac_max_pen:.0%} of folds at max penalty - registration still active.')


=== amplitude  (50 outer folds) ===
  1-SE penalty : {0.01: 1, 0.05: 9, 0.1: 21, 0.5: 19}
  1-SE n_basis : {30: 28, 40: 8, 60: 12, 80: 2}
  argmax penalty (what run 1 would have picked): {0.01: 14, 0.05: 19, 0.1: 15, 0.5: 2}
  backed off from argmax: 36/50 folds
  OK: only 38% of folds at max penalty - registration still active.

=== phase  (50 outer folds) ===
  1-SE penalty : {0.01: 6, 0.05: 7, 0.1: 22, 0.5: 15}
  1-SE n_basis : {30: 2, 40: 4, 60: 22, 80: 22}
  argmax penalty (what run 1 would have picked): {0.01: 17, 0.05: 17, 0.1: 14, 0.5: 2}
  backed off from argmax: 31/50 folds
  OK: only 30% of folds at max penalty - registration still active.

=== both  (50 outer folds) ===
  1-SE penalty : {0.01: 2, 0.05: 13, 0.1: 21, 0.5: 14}
  1-SE n_basis : {30: 19, 40: 12, 60: 15, 80: 4}
  argmax penalty (what run 1 would have picked): {0.01: 7, 0.05: 29, 0.1: 10, 0.5: 4}
  backed off from argmax: 30/50 folds
  OK: only 28% of folds at max penalty - registration still active.
